# 大阪カフェ出店診断 - 分析ノートブック

大阪府内のカフェ・スイーツ店舗データをホットペッパーAPIから取得し、
K-Meansクラスタリングで店舗タイプを分類するまでの分析プロセスです。


## 0. ライブラリのインポート

In [ ]:
!pip install japanize-matplotlib

In [ ]:
import requests
import pandas as pd
import numpy as np
import re
import time
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import japanize_matplotlib
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from scipy.spatial.distance import cdist

## 1. データ取得

ホットペッパーAPIは1リクエストあたりの上限1,000件のため、
middle_areaごとに分割して取得することで全件取得を実現しています。

In [ ]:
API_KEY = "YOUR_API_KEY"  # 環境変数での管理を推奨

gourmet_url = "http://webservice.recruit.co.jp/hotpepper/gourmet/v1/"
middle_area_url = "https://webservice.recruit.co.jp/hotpepper/middle_area/v1/"

# 大阪のmiddle_area一覧取得
params = {
    "key": API_KEY,
    "large_area": "Z023",  # 大阪府
    "format": "json"
}

response = requests.get(middle_area_url, params=params)
middle_areas = response.json()["results"]["middle_area"]

all_rows = []

for area in middle_areas:

    area_code = area["code"]
    area_name = area["name"]

    # まず件数取得
    params = {
        "key": API_KEY,
        "genre": "G014",
        "middle_area": area_code,
        "format": "json",
        "count": 1
    }

    total_count = int(
        requests.get(gourmet_url, params=params)
        .json()["results"]["results_available"]
    )

    if total_count == 0:
        continue

    # 100件ごとに取得
    for start in range(1, total_count + 1, 100):

        params = {
            "key": API_KEY,
            "genre": "G014",
            "middle_area": area_code,
            "format": "json",
            "count": 100,
            "start": start
        }

        shops = (
            requests.get(gourmet_url, params=params)
            .json()["results"]["shop"]
        )

        for shop in shops:
            all_rows.append({
                "id": shop.get("id"),
                "name": shop.get("name"),
                "station_name": shop.get("station_name"),
                "small_area": shop.get("small_area", {}).get("name"),
                "middle_area": shop.get("middle_area", {}).get("name"),
                "lat": shop.get("lat"),
                "lng": shop.get("lng"),
                "mobile_access": shop.get("mobile_access"),
                "budget.average": shop.get("budget", {}).get("average"),
                "capacity": shop.get("capacity"),
                "wifi": shop.get("wifi"),
                "card": shop.get("card"),
                "parking": shop.get("parking"),
                "non_smoking": shop.get("non_smoking"),
                "child": shop.get("child"),
                "open": shop.get("open"),
                "lunch": shop.get("lunch"),
                "midnight": shop.get("midnight")
            })

        time.sleep(0.2)  # API負荷軽減

df_all = pd.DataFrame(all_rows).drop_duplicates(subset="id")

print("取得件数:", len(df_all))

df_all.to_csv("osaka_cafe_raw.csv", index=False, encoding="utf-8-sig")
print("CSV保存完了")

## 2. データ確認

In [ ]:
df = pd.read_csv("osaka_cafe_raw.csv")
print(df.shape)
df.info()

In [ ]:
df.head()

## 3. クレンジング

分析対象として不適切な店舗（ネットカフェ・カラオケ・シーシャバー等）を除外します。

In [ ]:
# 不適切な店舗を名称から除外
exclude_words = "ネット|メディア|コミック|カラオケ|DiCE|KKラウンジ|シーシャ|しーしゃ|shisha|C.S.B|ムッシュ"

df = df[~df["name"].str.contains(exclude_words, case=False, na=False)]

print("クレンジング後件数:", len(df))

In [ ]:
# フードコート特定：同エリア・同席数の重複店舗を確認・除外
large_df = df[df["capacity"] >= 41]

dup_groups = (
    large_df
    .groupby(["small_area", "capacity"])
    .size()
    .reset_index(name="count")
)
dup_groups = dup_groups[dup_groups["count"] >= 2]

fcourt_df = large_df.merge(
    dup_groups[["small_area", "capacity"]],
    on=["small_area", "capacity"],
    how="inner"
)

# フードコートと判断したidを除外
remove_id = [
    "J000973640",
    "J000973646",
    "J000973674",
    "J001268101",
    "J001267504"
]

df = df[~df["id"].isin(remove_id)]

print("フードコート除外後件数:", len(df))

## 4. 特徴量作成

### 4-1. 徒歩時間の抽出

アクセス情報（mobile_access）から徒歩時間を正規表現で抽出します。
表記の揺れが多いため複数パターンに対応しています。
なお歩行速度は不動産広告のルールとして公正取引委員会と消費者庁が定めている分速80mを参考にしました。

In [ ]:
def extract_walk_minutes(text):

    if pd.isnull(text):
        return None

    match = re.search(r"徒歩約?(\d+)分|(\d+)分", text)
    if match:
        return int(match.group(1) or match.group(2))

    match = re.search(r"(\d+)秒", text)
    if match:
        return max(1, round(int(match.group(1)) / 60))

    match = re.search(r"(\d+(?:\.\d+)?)km", text)
    if match:
        return max(1, round(float(match.group(1)) * 1000 / 80))

    match = re.search(r"(\d+)(?:m|ﾒｰﾄﾙ|メートル)", text)
    if match:
        return max(1, round(int(match.group(1)) / 80))

    if re.search(r"すぐ|直結", text):
        return 1

    return None

df["walk_minutes"] = df["mobile_access"].apply(extract_walk_minutes)

### 4-2. 定休日の抽出

営業時間（open）の表記に一定の規則性があることを確認し、
正規表現によるルールベース処理で平日定休日・土日定休日を抽出します。
祝日・祝前日の表記は曜日判断のノイズとなるため除外しています。

In [ ]:
WEEKDAYS = ["月", "火", "水", "木", "金", "土", "日"]

def create_closed_flags(text):

    if pd.isna(text):
        return pd.Series([None, None])

    text = text.replace("祝日", "").replace("祝前日", "")

    open_days = set()

    # 月～水 のような範囲表現を抽出
    for start, end in re.findall(r'([月火水木金土日])～([月火水木金土日])', text):
        start_idx = WEEKDAYS.index(start)
        end_idx = WEEKDAYS.index(end)
        open_days.update(WEEKDAYS[start_idx:end_idx + 1])

    # 単独曜日を抽出
    open_days.update(re.findall(r'[月火水木金土日]', text))

    closed_days = set(WEEKDAYS) - open_days

    return pd.Series([
        int(len(closed_days & {"月", "火", "水", "木", "金"}) > 0),
        int(len(closed_days & {"土", "日"}) > 0)
    ])

df[["style_weekday_off", "style_weekend_off"]] = df["open"].apply(create_closed_flags)

### 4-3. 駐車場・子供可の抽出

In [ ]:
# 駐車場：コインパーキングは店側の設備ではないため除外した上で判定
df["parking_flag"] = np.where(
    df["parking"].str.contains("コイン", na=False),
    0,
    df["parking"].str.contains("あり|有", na=False).astype(int)
)

# 子供可：歓迎・OK・可 かつ 不可・NG でないものを1
# 「子供連れを歓迎している店舗」は店舗コンセプトに関わる特徴と判断
df["child_flag"] = (
    df["child"].str.contains("歓迎|OK|可", na=False)
    & ~df["child"].str.contains("不可|NG", na=False)
).astype(int)

### 4-4. その他の特徴量

In [ ]:
# Wi-Fi
wifi_map = {"あり": 1, "なし": 0}
df["wifi_flag"] = df["wifi"].map(wifi_map)

# カード決済
card_map = {"利用可": 1, "利用不可": 0}
df["card_flag"] = df["card"].map(card_map)

# 禁煙方針
smoking_map = {"禁煙席なし": 0, "一部禁煙": 1, "全面禁煙": 2}
df["non_smoking_score"] = df["non_smoking"].map(smoking_map)

# ランチ提供
lunch_map = {"あり": 1, "なし": 0}
df["lunch_flag"] = df["lunch"].map(lunch_map)

## 5. 欠損処理・最終データ確認

In [ ]:
# 席数・徒歩時間の欠損を除外
df = df.dropna(subset=["capacity", "walk_minutes"])

# Wi-Fi未確認 → なし（掲載されている場合は記載される可能性が高いため）
df["wifi_flag"] = df["wifi_flag"].fillna(0)

# 手動補完（Web検索で確認済み）
# 禁煙方針が未確認だった店舗
non_smoking_ids = [
    "J001225401", "J001145793", "J000865990", "J001053258",
    "J001230634", "J001074825", "J001224220"
]
df.loc[df["id"].isin(non_smoking_ids), "non_smoking_score"] = 2

# ランチ提供が未確認だった店舗
lunch_0_ids = ["J003297862", "J001294186"]
lunch_1_ids = ["J001263074", "J001222974", "J004067233", "J001268504", "J004477000"]
df.loc[df["id"].isin(lunch_0_ids), "lunch_flag"] = 0
df.loc[df["id"].isin(lunch_1_ids), "lunch_flag"] = 1

# 型変換
df["wifi_flag"] = df["wifi_flag"].astype(int)
df["card_flag"] = df["card_flag"].astype(int)
df["non_smoking_score"] = df["non_smoking_score"].astype(int)
df["lunch_flag"] = df["lunch_flag"].astype(int)
df["style_weekday_off"] = df["style_weekday_off"].astype(int)
df["style_weekend_off"] = df["style_weekend_off"].astype(int)

print("最終データ件数:", len(df))
print(df.isna().sum())

In [ ]:
# エリア供給密度（エリア内店舗数を供給密度指標として数値化）
area_counts = df["middle_area"].value_counts()
df["area_store_count"] = df["middle_area"].map(area_counts)

In [ ]:
# クラスタリングに使用する特徴量のみ表示
display_cols = [
    "capacity", "walk_minutes", "area_store_count",
    "wifi_flag", "card_flag", "parking_flag",
    "non_smoking_score", "child_flag",
    "style_weekday_off", "style_weekend_off", "lunch_flag"
]

df[display_cols].describe().round(2)

## 6. クラスタリング

In [ ]:
# ※ 再現性確保のため前処理済みデータを使用
# セクション1〜5で作成したosaka_cafe_preprocessed.csvを読み込む
# APIを再実行する場合はセクション1〜5を実行後、このセルをスキップしてください
df = pd.read_csv("osaka_cafe_preprocessed.csv")

In [ ]:
features = [
    # 立地・規模
    "capacity",
    "walk_minutes",
    "area_store_count",
    # 設備環境
    "wifi_flag",
    "card_flag",
    "parking_flag",
    "non_smoking_score",
    "child_flag",
    # 営業形態
    "style_weekday_off",
    "style_weekend_off",
    "lunch_flag"
]

X = df[features].copy()

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("kmeans", KMeans(n_clusters=5, random_state=42))
])

df["cluster"] = pipeline.fit_predict(X)

print(df["cluster"].value_counts().sort_index())

## 7. 可視化・根拠づけ

### 7-1. クラスタ別データ数

In [ ]:
cluster_count = df["cluster"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.bar(
    [f"クラスタ{i}" for i in cluster_count.index],
    cluster_count.values,
    color="steelblue"
)

for bar, count in zip(bars, cluster_count.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 5,
        f"{count}件",
        ha="center",
        fontsize=10
    )

ax.set_title("クラスタ別データ数", fontsize=14)
ax.set_xlabel("")
ax.set_ylabel("店舗数")
ax.set_ylim(0, cluster_count.max() + 40)

plt.tight_layout()
plt.savefig("cluster_count.png", dpi=150)
plt.show()

### 7-2. クラスタサマリー

In [ ]:
feature_names_jp = {
    "capacity": "席数",
    "walk_minutes": "徒歩時間",
    "area_store_count": "エリア供給密度",
    "wifi_flag": "Wi-Fi",
    "card_flag": "カード決済",
    "parking_flag": "駐車場",
    "non_smoking_score": "禁煙情報",
    "child_flag": "子供可",
    "style_weekday_off": "平日定休日",
    "style_weekend_off": "土日定休日",
    "lunch_flag": "ランチ提供"
}

cluster_summary = (
    df.groupby("cluster")[features]
    .mean()
    .round(2)
)

cluster_summary.index = [f"クラスタ{i}" for i in cluster_summary.index]
cluster_summary.columns = [feature_names_jp[col] for col in cluster_summary.columns]

display(cluster_summary)

### 7-3. クラスタ別・標準化後の特徴量ヒートマップ

In [ ]:
kmeans = pipeline.named_steps["kmeans"]

feature_names = list(feature_names_jp.values())

centers_df = pd.DataFrame(
    kmeans.cluster_centers_,
    columns=feature_names,
    index=[f"クラスタ{i}" for i in range(5)]
)

fig, ax = plt.subplots(figsize=(14, 6))

sns.heatmap(
    centers_df,
    annot=True,
    fmt=".2f",
    cmap="RdYlGn",
    center=0,
    linewidths=0.5,
    ax=ax
)

ax.set_title("クラスタ別・標準化後の特徴量中心値", fontsize=14, pad=12)
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=10)
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right', fontsize=10)

plt.tight_layout()
plt.savefig("cluster_feature_heatmap.png", dpi=150)
plt.show()

### 7-4. クラスタ中心点のPCA投影

In [ ]:
centers = kmeans.cluster_centers_

pca = PCA(n_components=2)
centers_2d = pca.fit_transform(centers)

colors = ["steelblue", "coral", "mediumseagreen", "mediumpurple", "tomato"]

fig, ax = plt.subplots(figsize=(8, 6))

for i, (x, y) in enumerate(centers_2d):
    ax.scatter(x, y, s=200, color=colors[i], zorder=3)
    ax.annotate(
        f"クラスタ{i}",
        (x, y),
        textcoords="offset points",
        xytext=(10, 5),
        fontsize=10
    )

ax.axhline(0, color="gray", linewidth=0.5, linestyle="--")
ax.axvline(0, color="gray", linewidth=0.5, linestyle="--")
ax.set_xlabel(f"PC1（寄与率: {pca.explained_variance_ratio_[0]:.1%}）")
ax.set_ylabel(f"PC2（寄与率: {pca.explained_variance_ratio_[1]:.1%}）")
ax.set_title("クラスタ中心点のPCA投影")

plt.tight_layout()
plt.savefig("cluster_centers_pca.png", dpi=150)
plt.show()

print(f"PC1寄与率: {pca.explained_variance_ratio_[0]:.1%}")
print(f"PC2寄与率: {pca.explained_variance_ratio_[1]:.1%}")
print(f"累積寄与率: {sum(pca.explained_variance_ratio_):.1%}")

### 7-5. クラスタ間距離ヒートマップ

In [ ]:
dist_matrix = cdist(centers, centers, metric='euclidean')

labels = [f"クラスタ{i}" for i in range(5)]
dist_df = pd.DataFrame(dist_matrix, index=labels, columns=labels)

fig, ax = plt.subplots(figsize=(8, 6))

sns.heatmap(
    dist_df,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    linewidths=0.5,
    ax=ax,
    cbar_kws={"label": "ユークリッド距離"}
)

ax.set_title("クラスタ間距離ヒートマップ", fontsize=14, pad=12)
ax.set_xlabel("")
ax.set_ylabel("")
plt.xticks(rotation=30, ha='right')
plt.yticks(rotation=0)

plt.tight_layout()
plt.savefig("cluster_distance_heatmap.png", dpi=150)
plt.show()

## 8. モデル・データの保存

In [ ]:
# パイプライン保存
joblib.dump(pipeline, "cafe_pipeline.pkl")

# エリア別クラスタ構成比（アプリのエリア傾向表示で使用）
cross_ratio = pd.crosstab(
    df["middle_area"],
    df["cluster"],
    normalize="index"
).round(2)
cross_ratio.to_csv("cluster_area_ratio.csv")

# エリアマッピング（アプリのエリア選択で使用）
area_mapping = df.groupby("middle_area").size().to_dict()
joblib.dump(area_mapping, "area_mapping.pkl")

# クラスタサマリー
cluster_summary.to_csv("cluster_summary.csv")

# クラスタリング済みデータ
df.to_csv("osaka_cafe_cluster.csv", index=False, encoding="utf-8-sig")

print("保存完了")